In [1]:
import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import anndata as ad

In [2]:
def split_taxonomy_and_save(data, exceptions_path):    
    # Split the taxa names by "." and count the maximum number of levels
    taxa_split = data.var['taxa'].str.split('.')
    print(taxa_split.apply(len).value_counts().sort_index())
    
    # exceptions located in the period_taxa.csv file
    period_taxa = pd.read_csv(exceptions_path, header=None)
    period_taxa = period_taxa.drop(index=0)

    # Merge entries in each row with a '.' separator
    # Drop the first row and merge entries in each row with a '.' separator
    merged_taxa = period_taxa.apply(lambda row: '.'.join(row.dropna().astype(str)), axis=1)
    # Create a DataFrame with merged_taxa as the index and period_taxa as the data
    period_taxa.index = merged_taxa
    
    # Define a function to assign categories based on the period_taxa DataFrame
    def assign_categories(taxa_name, period_taxa):
        split_taxa = taxa_name.split(".")
        if len(split_taxa) > 6:
            # Use the period_taxa Series to assign categories properly
            if taxa_name not in period_taxa.index:
                raise ValueError(f"Taxa '{taxa_name}' not found in period_taxa.")
            return period_taxa.loc[taxa_name]
        else:
            # Assign generic categories for taxa with 6 or fewer levels
            return split_taxa

    # Apply the function to the taxa names and create a new varm
    taxon_lists = data.var["taxa"].apply(assign_categories, period_taxa=period_taxa)
    categories = ["Domain", "Phylum", "Class", "Order", "Family", "Genus"]

    taxon_df = pd.DataFrame(taxon_lists.tolist(), index=data.var_names, columns=categories)

    return taxon_df

In [3]:
DATA_DIR = "../../datasets/up_to_date_sep10/"

data = ad.read_h5ad(DATA_DIR + "pretrain.h5ad")
taxon_df = split_taxonomy_and_save(data, DATA_DIR + "period_taxa.csv")

data.varm['taxonomy'] = taxon_df
data.write_h5ad(DATA_DIR + "pretrain.h5ad")


taxa
6    4655
7      20
8       5
Name: count, dtype: int64


In [7]:
data.varm['taxonomy']

,Domain,Phylum,Class,Order,Family,Genus
Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Atopobiaceae.Tractidigestivibacter,Bacteria,Actinomycetota,Coriobacteriia,Coriobacteriales,Atopobiaceae,Tractidigestivibacter
Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Coriobacteriaceae.Collinsella,Bacteria,Actinomycetota,Coriobacteriia,Coriobacteriales,Coriobacteriaceae,Collinsella
Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Eggerthellaceae.Adlercreutzia,Bacteria,Actinomycetota,Coriobacteriia,Coriobacteriales,Eggerthellaceae,Adlercreutzia
Bacteria.Actinomycetota.Coriobacteriia.Coriobacteriales.Eggerthellaceae.Senegalimassilia,Bacteria,Actinomycetota,Coriobacteriia,Coriobacteriales,Eggerthellaceae,Senegalimassilia
Bacteria.Bacillota.Bacilli.Erysipelotrichales.Erysipelotrichaceae.Holdemanella,Bacteria,Bacillota,Bacilli,Erysipelotrichales,Erysipelotrichaceae,Holdemanella
...,...,...,...,...,...,...
Bacteria.Bacteroidota.Bacteroidia.Flavobacteriales.Flavobacteriaceae.Flavirhabdus,Bacteria,Bacteroidota,Bacteroidia,Flavobacteriales,Flavobacteriaceae,Flavirhabdus
Bacteria.Pseudomonadota.Alphaproteobacteria.Rhodobacterales.Paracoccaceae.Octadecabacter,Bacteria,Pseudomonadota,Alphaproteobacteria,Rhodobacterales,Paracoccaceae,Octadecabacter
Bacteria.Pseudomonadota.Alphaproteobacteria.Acetobacterales.Acetobacteraceae.Swingsia,Bacteria,Pseudomonadota,Alphaproteobacteria,Acetobacterales,Acetobacteraceae,Swingsia
Bacteria.Bacteroidota.Bacteroidia.Flavobacteriales.Flavobacteriaceae.Aurantiacicella,Bacteria,Bacteroidota,Bacteroidia,Flavobacteriales,Flavobacteriaceae,Aurantiacicella
